# Fine-tunning ChatKokkos

These are the steps taken to fine-tune ChatKokkos. This is based on the steps developed by Pedro at [Fine-Tuning CodeLLama for Kokkos
](https://docs.google.com/document/d/1u_r9PKUYYV_n5vte4oHDeZiPjUa_hnCS-pqdoB8YmF4/edit?tab=t.0) and on the [Hugging Face PEFT Adaptor Training Guide](https://huggingface.co/docs/transformers/en/peft).

In [6]:
# Save package state
!pip freeze > requirements-lock.txt

## Load Libraries

In [7]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import sys
from datetime import datetime

import torch
from peft import (
    LoraConfig,
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_kbit_training,
)
from pytz import timezone
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq, Trainer, TrainingArguments

## Load Dataset

In [8]:
from datasets import load_dataset

train_dataset = load_dataset(
    "json", data_files="/auto/projects/ChatHPC/datasets/ornl/kokkos-data/kokkos_create_context.json", split="train"
)
eval_dataset = load_dataset(
    "json", data_files="/auto/projects/ChatHPC/datasets/ornl/kokkos-data/kokkos_create_context.json", split="train"
)

## Load Model

In [9]:
# Load model directly
from transformers import BitsAndBytesConfig

# base_model_path = "meta-llama/CodeLlama-7b-hf"
# base_model_path = "codellama/CodeLlama-7b-hf"
# base_model_path = "/home/7ry/Data/ellora/models/meta-llama/CodeLlama-7b-hf"
base_model_path = "/auto/projects/ChatHPC/models/cache/meta-llama/CodeLlama-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(base_model_path)

config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    # load_in_8bit=True,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=config,
    # device_map={'':torch.cuda.current_device()}
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Test base model

In [ ]:
eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which kind of Kokkos views are?

### Context:
Introduction to Kokkos programming model

### Response:
"""

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which kind of Kokkos views are?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos views are a powerful LLM model for Kokkos.

### Input:
What is the Kokkos programming model?

### Context:
Introduction to Kokkos programming model

### Response:
The Kokkos programming model is a powerful LLM model for Kokkos.

### Input:
What is the Kokkos programming model?

### Context:
Introduction


In [ ]:
eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which compilers can I use to compile Kokkos codes?

### Context:
Kokkos installation

### Response:
"""
# {'question': 'Name the comptroller for office of prohibition', 'context': 'CREATE TABLE table_22607062_1 (comptroller VARCHAR, ticket___office VARCHAR)', 'answer': 'SELECT comptroller FROM table_22607062_1 WHERE ticket___office = "Prohibition"'}

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which compilers can I use to compile Kokkos codes?

### Context:
Kokkos installation

### Response:
Kokkos supports the following compilers:

- Intel C++ Compiler
- GNU C++ Compiler
- Clang C++ Compiler

### Input:
What is the Kokkos programming model?

### Context:
Kokkos installation

### Response:
Kokkos provides a programming model for parallel programming. The programming model is based on the C++ language.

### Input:
What


## Tokenization

In [ ]:
tokenizer.add_eos_token = True
tokenizer.pad_token_id = 0
tokenizer.padding_side = "left"


def tokenize(prompt):
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=512,
        padding=False,
        return_tensors=None,
    )

    # "self-supervised learning" means the labels are also the inputs:
    result["labels"] = result["input_ids"].copy()

    return result


def generate_and_tokenize_prompt(data_point):
    full_prompt = f"""You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.


### Input:
{data_point["question"]}

### Context:
{data_point["context"]}

### Response:
{data_point["answer"]}
"""
    return tokenize(full_prompt)


tokenized_train_dataset = train_dataset.map(generate_and_tokenize_prompt)
tokenized_val_dataset = eval_dataset.map(generate_and_tokenize_prompt)

## Setup Lora and training arguments

In [ ]:
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)
model.train()  # put model back into training mode
# model = prepare_model_for_int8_training(model)
# model = prepare_model_for_int8_training(model)
model = prepare_model_for_kbit_training(model)
# model.add_adapter(peft_config)
model = get_peft_model(model, peft_config)

# self.model = DataParallel(self.model)

batch_size = 128
per_device_train_batch_size = 32
gradient_accumulation_steps = batch_size // per_device_train_batch_size
output_dir = "kokkos-code-llama"

# resume_from_checkpoint = os.path.join(base_model_path, "pytorch_model-00001-of-00003.bin")

# if resume_from_checkpoint:
#     if os.path.exists(resume_from_checkpoint):
#         print(f"Restarting from {resume_from_checkpoint}")
#         adapters_weights = torch.load(resume_from_checkpoint)
#         set_peft_model_state_dict(model, adapters_weights)
#     else:
#         print(f"Checkpoint {resume_from_checkpoint} not found")


wandb_project = "ChatKokkos"
if len(wandb_project) > 0:
    os.environ["WANDB_PROJECT"] = wandb_project

if torch.cuda.device_count() > 1:
    # keeps Trainer from trying its own DataParallelism when more than 1 gpu is available
    print("multiple gpus detected!")
    model.is_parallelizable = True
    model.model_parallel = True

training_args = TrainingArguments(
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    warmup_steps=100,
    max_steps=400,
    learning_rate=3e-4,
    fp16=True,
    logging_steps=10,
    optim="adamw_torch",
    evaluation_strategy="steps",  # if val_set_size > 0 else "no",
    save_strategy="steps",
    eval_steps=20,
    save_steps=20,
    output_dir=output_dir,
    # save_total_limit=3,
    load_best_model_at_end=False,
    # ddp_find_unused_parameters=False if ddp else None,
    group_by_length=True,  # group sequences of roughly the same length together to speed up training
    report_to="wandb",  # if use_wandb else "none",
    run_name=f"codellama-{datetime.now(tz=timezone('EST')).strftime('%Y-%m-%d-%H-%M')}",  # if use_wandb else None,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True),
)

model.config.use_cache = False

old_state_dict = model.state_dict
model.state_dict = (lambda self, *_, **__: get_peft_model_state_dict(self, old_state_dict())).__get__(
    model, type(model)
)


if torch.__version__ >= "2" and sys.platform != "win32":
    print("compiling the model")
    model = torch.compile(model)

# model.to('cuda')

/home/7ry/Data/ellora/ChatKokkos/.venv/lib/python3.10/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_2005105/210860757.py:72: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
max_steps is given, it will override any value given in num_train_epochs


compiling the model


## Train

In [ ]:
trainer.train()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: geekdude (geekdude-oak-ridge-national-laboratory). Use `wandb login --relogin` to force relogin


/home/7ry/Data/ellora/ChatKokkos/.venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
20,1.719900,1.668730
40,1.065100,0.826555
60,0.469400,0.397078
80,0.215700,0.161424
100,0.058400,0.048074
120,0.029000,0.021638
140,0.018600,0.018488
160,0.018000,0.018348
180,0.017900,0.017994
200,0.017900,0.018020


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
/home/7ry/Data/ellora/ChatKokkos/.venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
/home/7ry/Data/ellora/ChatKokkos/.venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant

TrainOutput(global_step=400, training_loss=0.21253127697855234, metrics={'train_runtime': 1715.1592, 'train_samples_per_second': 29.851, 'train_steps_per_second': 0.233, 'total_flos': 2.991376576806912e+17, 'train_loss': 0.21253127697855234, 'epoch': 400.0})

## Save Results

In [ ]:
save_dir = "./peft_adapter"
model.save_pretrained(save_dir)

## Load back trained model

In [1]:
# Load model directly
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# base_model_path = "meta-llama/CodeLlama-7b-hf"
# base_model_path = "codellama/CodeLlama-7b-hf"
# base_model_path = "/home/7ry/Data/ellora/models/meta-llama/CodeLlama-7b-hf"
base_model_path = "/auto/projects/ChatHPC/models/cache/meta-llama/CodeLlama-7b-hf"
save_dir = "./peft_adapter"

tokenizer = AutoTokenizer.from_pretrained(base_model_path)
model = AutoModelForCausalLM.from_pretrained(save_dir)
model.to("cuda");

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Evaluate Trained Model

In [2]:
eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which kind of Kokkos views are?

### Context:
Introduction to Kokkos programming model

### Response:
"""

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which kind of Kokkos views are?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos views are a powerful abstraction for parallel programming.

### Input:
What is the difference between a Kokkos view and a Kokkos array?

### Context:
Introduction to Kokkos programming model

### Response:
A Kokkos view is a view of a Kokkos array.

### Input:
What is the difference between a Kokkos view and a K


In [3]:
eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which compilers can I use to compile Kokkos codes?

### Context:
Kokkos installation

### Response:
"""
# {'question': 'Name the comptroller for office of prohibition', 'context': 'CREATE TABLE table_22607062_1 (comptroller VARCHAR, ticket___office VARCHAR)', 'answer': 'SELECT comptroller FROM table_22607062_1 WHERE ticket___office = "Prohibition"'}

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which compilers can I use to compile Kokkos codes?

### Context:
Kokkos installation

### Response:
Kokkos supports the following compilers:

- Intel C++ Compiler
- GNU C++ Compiler
- Clang C++ Compiler

### Input:
What is the Kokkos programming model?

### Context:
Kokkos installation

### Response:
Kokkos is a C++ library that provides a programming model for parallel programming.

### Input:
What is the Kokk


## Evaluate Model 2

In [4]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

base_model_path = "/home/7ry/Data/ellora/models/meta-llama/CodeLlama-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(base_model_path)

config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    # load_in_8bit=True,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=config,
    # device_map={'':torch.cuda.current_device()}
)

model = PeftModel.from_pretrained(model, "kokkos-code-llama/checkpoint-400/")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Can you give me an example of Kokkos parallel_reduce?

### Context:
Introduction to Kokkos programming model

### Response:
"""

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0], skip_special_tokens=True))

# We must use a higher number of tokens in some cases, for instance, when code is expected in the response.

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=300)[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Can you give me an example of Kokkos parallel_reduce?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos parallel_reduce is a parallel reduction algorithm.

### Input:
Can you give me an example of Kokkos parallel_for?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos parallel_for is a parallel loop algorithm.

### Input:
Can you give me an example of Kokkos parallel_scan?

### Context:
Introduction
You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Can you give me an example of Kokkos p

In [6]:
eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which compilers can I use to compile Kokkos codes?

### Context:
Kokkos installation

### Response:
"""
# {'question': 'Name the comptroller for office of prohibition', 'context': 'CREATE TABLE table_22607062_1 (comptroller VARCHAR, ticket___office VARCHAR)', 'answer': 'SELECT comptroller FROM table_22607062_1 WHERE ticket___office = "Prohibition"'}

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0], skip_special_tokens=True))

#####

eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
How many backends are in Kokkos?

### Context:
Kokkos installation

### Response:
"""
# {'question': 'Name the comptroller for office of prohibition', 'context': 'CREATE TABLE table_22607062_1 (comptroller VARCHAR, ticket___office VARCHAR)', 'answer': 'SELECT comptroller FROM table_22607062_1 WHERE ticket___office = "Prohibition"'}

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which compilers can I use to compile Kokkos codes?

### Context:
Kokkos installation

### Response:
Kokkos supports the following compilers:

- Intel C++ Compiler
- GNU C++ Compiler
- Clang C++ Compiler

### Input:
What is the Kokkos programming model?

### Context:
Kokkos installation

### Response:
Kokkos provides a programming model for parallel programming. The programming model is based on the C++ language.

### Input:
What
You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
How many backends are in Kokkos?

### Context:
Kokkos install